In [0]:
from pyspark.sql.functions import avg, col
import plotly.express as px

# Calculate average price per county and property type
avg_prices = spark.table("uk_property_insights.default.silver_property_prices") \
    .groupBy("county", "property_type") \
    .agg(avg("price").alias("avg_price")) \
    .filter(col("county").isNotNull())

# Find top 5 counties by overall average price
top_counties = spark.table("uk_property_insights.default.silver_property_prices") \
    .groupBy("county") \
    .agg(avg("price").alias("overall_avg_price")) \
    .filter(col("county").isNotNull()) \
    .orderBy(col("overall_avg_price").desc()) \
    .limit(5) \
    .select("county")

# Filter to only top 5 counties
top_5_data = avg_prices.join(top_counties, "county", "inner")

# Convert to pandas for visualization
df_viz = top_5_data.toPandas()

# Create grouped bar chart
fig = px.bar(
    df_viz.sort_values("avg_price", ascending=False),
    x="county",
    y="avg_price",
    color="property_type",
    title="Top 5 Counties by Average Property Price (Grouped by Property Type)",
    labels={"avg_price": "Average Price (£)", "county": "County", "property_type": "Property Type"},
    barmode="group",
    height=500
)

fig.update_layout(
    xaxis_title="County",
    yaxis_title="Average Price (£)",
    yaxis_tickformat=",.0f"
)

fig.show()

In [0]:
# Display the data with built-in visualization
display(top_5_data.orderBy(col("avg_price").desc()))

Databricks visualization. Run in Databricks to view.